In [39]:
# CONFIG — change SEASON and re-run to pull any year
# ============================================================
SEASON = 2026
 
# Season start (drops spring training). Update opener date per year if needed.
SEASON_START_DATES = {
    2025: '2025-03-27',
    2026: '2026-03-25',
}
# Statcast pull window (wide enough to catch the whole season; partial seasons ok)
PULL_WINDOWS = {
    2025: ('2025-03-01', '2025-10-01'),
    2026: ('2026-03-01', '2026-07-01'),
}
 
GS_MIN = 4                      # min games started to include a pitcher
PWIN = 5                        # pitcher rolling window (last N starts)
RECENT_YEAR = 2025             # for name-resolution recency disambiguation
 
GAMES_FEATURES_PATH = f'/Users/jackdiamond/Documents/Sports_Models/MLB/O/U/Data/games_{SEASON}_features.csv'
PITCHER_STARTS_OUT  = f'pitcher_starts_{SEASON}.csv'
GAMES_V2_OUT        = f'games_{SEASON}_features_v2.1.csv'
 
import pandas as pd
SEASON_START = pd.Timestamp(SEASON_START_DATES[SEASON])
PULL_START, PULL_END = PULL_WINDOWS[SEASON]

# 0. STARTER LIST — pitchers with >= GS_MIN starts this season
# ============================================================
from pybaseball import pitching_stats_bref
bref = pitching_stats_bref(SEASON)
starters = bref[bref['GS'] >= GS_MIN].copy()
print(f"Starters to pull ({SEASON}, GS>={GS_MIN}): {len(starters)}")

Starters to pull (2026, GS>=4): 202


In [40]:
import time
import unicodedata
from pybaseball import playerid_lookup
 
def fix_mojibake(name):
    """'Jos\\xc3\\xa9' -> 'José'. No-op if already clean."""
    try:
        return name.encode('latin-1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError, AttributeError):
        return name
 
def resolve_id(name):
    clean = fix_mojibake(name)
    parts = clean.strip().split()
    first, last = parts[0], parts[-1]
    lookup = playerid_lookup(last, first, fuzzy=True)
    if lookup.empty:
        return None, None
    lookup = lookup.copy()
    lookup['played_last_num'] = pd.to_numeric(lookup['mlb_played_last'], errors='coerce')
    active = lookup[lookup['played_last_num'] >= RECENT_YEAR]
    row = active.iloc[0] if not active.empty else lookup.iloc[0]
    return int(row['key_mlbam']), f"{row['name_first']} {row['name_last']}"
 
names = starters['Name'].unique().tolist()
print(f"Resolving {len(names)} names...")
 
id_map, failed, review = {}, [], []
for name in names:
    try:
        pid, matched = resolve_id(name)
        if pid is None:
            failed.append(name)
        else:
            id_map[name] = pid
            review.append((name, matched))
    except Exception:
        failed.append(name)
    time.sleep(0.1)
 
print(f"Resolved: {len(id_map)} | Failed: {len(failed)}")
print("Still failed:", failed)
 
print("\n--- Matches to review (last-name mismatch) ---")
for original, matched in review:
    o = fix_mojibake(original).lower().split()[-1][:4]
    m = matched.lower().split()[-1][:4]
    if o != m:
        print(f"REVIEW: '{original}' -> '{matched}'")

Resolving 202 names...
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Returning the 5 most similar names.
No identically matched names found! Retur

In [45]:
from pybaseball import statcast_pitcher
 
SWING = ['swinging_strike','swinging_strike_blocked','foul','foul_tip','hit_into_play']
WHIFF = ['swinging_strike','swinging_strike_blocked']
 
def aggregate_starts(pitch_df, pitcher_name, mlbam_id):
    df = pitch_df.copy()
    df['is_swing'] = df['description'].isin(SWING)
    df['is_whiff'] = df['description'].isin(WHIFF)
    df['is_pa']    = df['events'].notna()
    df['pitcher_team'] = df['home_team'].where(df['inning_topbot'] == 'Top', df['away_team'])
 
    g = df.groupby(['game_date','game_pk']).agg(
        batters_faced=('is_pa','sum'),
        strikeouts=('events', lambda x: (x=='strikeout').sum()),
        walks=('events', lambda x: (x=='walk').sum()),
        swings=('is_swing','sum'),
        whiffs=('is_whiff','sum'),
        pitches=('description','size'),
        pitcher_team=('pitcher_team','first'),
        home_team=('home_team','first'),
        away_team=('away_team','first'),
    ).reset_index()
    g['pitcher_name'] = pitcher_name
    g['mlbam_id']  = mlbam_id
    g['K_pct']     = g['strikeouts'] / g['batters_faced']
    g['BB_pct']    = g['walks'] / g['batters_faced']
    g['whiff_pct'] = g['whiffs'] / g['swings'].replace(0, pd.NA)
    return g
 
all_starts, pull_failed = [], []
for i, (name, pid) in enumerate(id_map.items(), 1):
    try:
        pitches = statcast_pitcher(PULL_START, PULL_END, pid)
        if pitches.empty:
            pull_failed.append(name); continue
        all_starts.append(aggregate_starts(pitches, name, pid))
    except Exception:
        pull_failed.append(name)
    if i % 25 == 0:
        print(f"  ...{i}/{len(id_map)} pulled")
    time.sleep(0.5)
 
pitcher_starts = pd.concat(all_starts, ignore_index=True)
pitcher_starts['game_date'] = pd.to_datetime(pitcher_starts['game_date'])
pitcher_starts = pitcher_starts[pitcher_starts['game_date'] >= SEASON_START].reset_index(drop=True)
 
pitcher_starts.to_csv(PITCHER_STARTS_OUT, index=False)
print(f"\nDone. {len(pitcher_starts)} regular-season start-lines from "
      f"{pitcher_starts['mlbam_id'].nunique()} pitchers. Failed: {len(pull_failed)}")

Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
  ...25/202 pulled
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Player Data
Gathering Pla

In [52]:
games = pd.read_csv('O/U/Data/games_2025_features.csv')

# Slim the pitcher features we want to attach
roll_cols = ['K_pct_roll5', 'BB_pct_roll5', 'whiff_pct_roll5']
sp = pitcher_starts[['game_date', 'pitcher_team', 'pitcher_name'] + roll_cols].copy()

# Your games frame needs a datetime game_date to match on
games['game_date'] = pd.to_datetime(games['Date'])

# --- Join 1: HOME starter (pitcher_team == home_team) ---
home_sp = sp.rename(columns={c: f'home_SP_{c}' for c in roll_cols})
home_sp = home_sp.rename(columns={'pitcher_name': 'home_SP_name'})
games = games.merge(
    home_sp.drop(columns='pitcher_team').assign(_t=home_sp['pitcher_team']),
    left_on=['game_date', 'home_team'],
    right_on=['game_date', '_t'],
    how='left'
).drop(columns='_t')

# --- Join 2: AWAY starter (pitcher_team == away_team) ---
away_sp = sp.rename(columns={c: f'away_SP_{c}' for c in roll_cols})
away_sp = away_sp.rename(columns={'pitcher_name': 'away_SP_name'})
games = games.merge(
    away_sp.drop(columns='pitcher_team').assign(_t=away_sp['pitcher_team']),
    left_on=['game_date', 'away_team'],
    right_on=['game_date', '_t'],
    how='left'
).drop(columns='_t')

# Dedupe any doubleheader-induced duplicates
games = games.drop_duplicates(subset=['game_date','home_team','away_team'], keep='first').reset_index(drop=True)

print(f"Games: {len(games)}")
print(f"Games with home SP matched: {games['home_SP_name'].notna().sum()}")
print(f"Games with away SP matched: {games['away_SP_name'].notna().sum()}")
print(games[['game_date','home_team','away_team','home_SP_name',
             'home_SP_K_pct_roll5','away_SP_name','away_SP_K_pct_roll5']].head())

ValueError: You are trying to merge on datetime64[ns] and object columns for key 'game_date'. If you wish to proceed you should use pd.concat